# Microservices & Resilience

## What's covered

- **Monolith vs microservices** — what you actually trade
- **API Gateway** — the single entry point that hides service-fleet complexity from clients
- **Service mesh** — the sidecar pattern for service-to-service traffic
- **The Saga pattern** — managing distributed transactions without two-phase commit
- **Circuit breaker** — failing fast when a downstream is broken
- **Bulkhead** — isolating failures so they don't take everything down
- **Idempotency and safe retries** — the foundation under everything that retries


## Monolith vs microservices — the real trade-off

A **monolith** is one deployable unit running all of the application's code in one process. A **microservices** architecture splits the application into many small services, each independently deployed, communicating over the network.

The framing "monolith bad, microservices good" is wrong. Both are valid architectural choices that trade different costs.

**Monolith wins:**

- **One deployable.** Build, test, deploy is one pipeline. One process to operate. One log to read.
- **In-process calls are free.** No network hop, no serialization, no marshalling. Function call latency is nanoseconds.
- **Transactional consistency.** A single database transaction can span the whole codebase. Cross-module operations are atomic for free.
- **Easier refactoring.** Moving code between modules is just `git mv`. No coordination with other teams.
- **Easier debugging.** One stack trace. One process to attach a debugger to.

**Microservices wins:**

- **Independent deployment.** Teams can ship at their own cadence without waiting for the rest of the system.
- **Independent scaling.** The image-resize service can run on twenty machines while the user-profile service runs on two.
- **Technology diversity.** One service in Python for ML, another in Go for performance, another in Rust for safety. Each team picks the right tool.
- **Failure isolation.** A bug in the recommendation engine doesn't crash the checkout service.
- **Org alignment (Conway's law).** Services map to teams. Each team owns its services end-to-end.

**The microservices tax:**

- **Network failures everywhere.** Every cross-service call can fail, time out, or return a partial result. The application code has to handle this.
- **Distributed transactions.** What was one ACID transaction in a monolith is now a saga across services (covered below).
- **Observability is harder.** A user request fans out across ten services; following it requires distributed tracing.
- **Latency is higher.** Even sub-millisecond network calls add up; a chain of ten sequential calls can blow a 100ms budget.
- **Operational cost.** Each service needs its own deploy pipeline, monitoring, on-call rotation, runbook.

**The honest decision rule.** Start with a monolith. Split off a service when *the cost of keeping it in the monolith exceeds the cost of running it as a service*. That cost is usually about team independence — when two teams keep stepping on each other in the same codebase, splitting is a relief. It's almost never about technical scalability — a well-built monolith on modern hardware can handle tens of thousands of QPS.

**Conway's law** is the silent driver: *organizations design systems that mirror their communication structure*. A monolith works when one team owns it. Microservices work when each service has one team. When the team structure changes faster than the architecture catches up, the architecture starts fighting the org chart, and refactors are the consequence.


## API Gateway — one front door

When you have many backend services, the client (browser, mobile app, partner) shouldn't know about all of them. The **API gateway** is a single HTTPS endpoint that fronts the entire service fleet. Clients call the gateway; the gateway routes to the right backend.

**What the gateway does:**

- **Routing.** `/users/*` goes to the user service; `/orders/*` to the order service; etc.
- **Authentication & authorization.** Verify the JWT or session, attach the user identity to downstream calls, reject unauthorized requests at the edge.
- **Rate limiting.** Per-API-key, per-user, per-endpoint quotas enforced centrally.
- **Request/response transformation.** Hide backend protocols (rewrite a REST call into gRPC, translate JSON shapes).
- **TLS termination.** Backend services can speak plain HTTP behind the gateway.
- **Aggregation.** One gateway request can fan out to multiple backends and combine the results (common in Backend-for-Frontend patterns).
- **Versioning.** Route `/v1/users` and `/v2/users` to different backends.

**The trade-off.** The gateway is a critical single point of dependency. Every external request goes through it. It must be highly available, highly performant, and operated to a higher standard than any backend service. Pin too much logic in the gateway and it becomes a god service that no one wants to deploy. Pin too little and you lose the centralization benefits.

**Common implementations:** AWS API Gateway, Google Cloud Endpoints, Kong, Tyk, Envoy (with Istio config), NGINX, Cloudflare's gateway products.

**A useful split:** the gateway handles *concerns about how clients talk to you* (auth, rate limit, routing, TLS). Backend services handle *concerns about what your business does*. Anything that's business logic doesn't belong in the gateway.


## Service mesh — wiring service-to-service traffic

The API gateway sits between clients and your services. The **service mesh** sits between your services.

A service mesh installs a small proxy (a **sidecar**) alongside every service instance. All inter-service traffic flows through the sidecar instead of going directly. The sidecar handles cross-cutting concerns the same way for every service in the mesh, regardless of language:

- **Mutual TLS** between services without each service implementing TLS.
- **Retries, timeouts, circuit breakers** without each service implementing them.
- **Routing rules** (canary deployments, traffic splitting for experiments).
- **Observability** (metrics, traces, logs) emitted uniformly from every sidecar.
- **Service discovery** without service code knowing about it.

```
                                control plane
                                    |
   +----[svc A]---[sidecar A]---<---+--->---[sidecar B]---[svc B]
                       |                          |
                       +------- mTLS, retries, ---+
                               metrics, routing
                               (data plane)
```

**The control plane** (Istio, Linkerd's control plane, Consul Connect) distributes configuration to all the sidecars. **The data plane** (Envoy as the sidecar, in the case of Istio; Linkerd's own proxy in Linkerd's case) handles the actual traffic.

**The win:** application code stays simple. The service just makes plain HTTP calls; the sidecar enforces all the resilience policy, observability, security. Policies are uniform — no service can "forget" to retry with backoff, because the sidecar does it.

**The cost:** the mesh is a substantial operational system in its own right. Sidecars consume CPU and memory per pod. The control plane is a critical dependency. Many teams that adopted Istio in 2018-2020 ultimately rolled it back as the complexity outweighed the benefits.

**The honest take.** Service mesh earns its keep at *significant* scale (hundreds of services, multi-team org) where the alternative is bespoke libraries in every language. At smaller scale, a shared library or framework (a "thick client" pattern) often beats the mesh on operational simplicity.


## The Saga pattern — distributed transactions

In a monolith, multi-step business operations live in a single ACID transaction. Either all the steps commit, or all roll back. In microservices, each service owns its own database; cross-service ACID isn't possible without **two-phase commit** (XA), which has known availability problems and is rarely used in modern systems.

The **saga pattern** is the alternative: a multi-step operation is modeled as a sequence of *local* transactions in different services, with **compensating actions** for each step that can roll it back if a later step fails.

Worked example — placing an order:

```
   1. Order service:     create order               (compensate: cancel order)
   2. Inventory service: reserve items              (compensate: release reservation)
   3. Payment service:   charge card                (compensate: refund card)
   4. Shipping service:  schedule shipment          (compensate: cancel shipment)
```

If step 3 fails — the card is declined — the saga runs the compensating actions for steps 1 and 2 in reverse: release the reservation, cancel the order. The card was never charged, so step 3 has nothing to compensate.

**Two coordination styles:**

- **Orchestration.** A central coordinator (the "saga orchestrator") issues commands to each service in turn and listens for replies. On failure, the orchestrator runs the compensations. Easy to reason about; the workflow lives in one place.
- **Choreography.** No coordinator. Each service publishes events when it completes; other services subscribe and act. The order service publishes `OrderCreated`; the inventory service listens, reserves, and publishes `ItemsReserved`; etc. Looser coupling; harder to debug because the workflow is implicit.

**Sagas are not ACID.** They're *eventually consistent*. Between steps, the system can observe partial state — an order exists but isn't paid for yet, for a few seconds. The application has to be designed for this.

**Compensations are not undo.** They're a *new transaction* that conceptually reverses the effect of the original. You can't "uncharge" a credit card; you can issue a refund. Some things genuinely can't be compensated (an email already sent). Saga design has to acknowledge what's compensatable and what's not.

**The pragmatic rule:** if your operation truly requires ACID across services, you either need (a) to combine those services back into one, (b) to use distributed SQL (Spanner, CockroachDB) and keep them strongly consistent, or (c) to accept saga's eventual consistency and design the user experience around it. Two-phase commit across services is almost never the right answer in modern systems.


## Circuit Breaker — failing fast

When a downstream service is failing — timing out, returning errors, taking 10 seconds per call — the *worst* thing your service can do is keep calling it. Every call ties up a connection, a thread, a memory buffer. Soon your service is unhealthy too, not because it broke but because it's queued behind a broken dependency.

The **circuit breaker** pattern detects when a downstream is failing and **stops calling it** for a cooldown period. Calls fail immediately with a known error instead of timing out. This frees your service's resources and prevents cascading failure.

A circuit breaker has three states:

- **Closed** — normal. Calls pass through. Track success/failure rate.
- **Open** — the failure threshold has been exceeded. All calls fail immediately without touching the downstream. Wait for the cooldown period.
- **Half-open** — after cooldown, allow a few probe calls through. If they succeed, close the circuit. If they fail, open it again.

```
                 +-----------+
                 | CLOSED    |
   +-->----+-----| (normal)  |
   |       |     +-----------+
   |       |       | threshold reached
   |       v       v
   |   +-----------+
   |   |  OPEN     |
   |   | (fail fast)|
   |   +-----------+
   |       | cooldown elapsed
   |       v
   |   +-----------+
   |   | HALF-OPEN |
   +<--| (probing) |
       +-----------+
         | probe failed -> open
```

The threshold is usually defined by a window (last N calls or last T seconds). Common defaults: 50% error rate over a 10-second window opens the circuit; cooldown 30 seconds; 1-3 probe calls in half-open.


### Circuit breaker in code


In [ ]:
import time
from dataclasses import dataclass, field
from enum import Enum


class State(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"


@dataclass
class CircuitBreaker:
    failure_threshold: int = 3       # consecutive failures before opening
    cooldown_seconds: float = 1.0    # how long to stay open before half-open
    state: State = State.CLOSED
    consecutive_failures: int = 0
    opened_at: float = 0.0

    def call(self, fn, *args, **kwargs):
        if self.state == State.OPEN:
            if time.monotonic() - self.opened_at >= self.cooldown_seconds:
                self.state = State.HALF_OPEN
            else:
                raise RuntimeError("circuit OPEN — short-circuiting")

        try:
            result = fn(*args, **kwargs)
        except Exception:
            self._on_failure()
            raise
        else:
            self._on_success()
            return result

    def _on_failure(self):
        self.consecutive_failures += 1
        if self.consecutive_failures >= self.failure_threshold:
            self.state = State.OPEN
            self.opened_at = time.monotonic()

    def _on_success(self):
        self.consecutive_failures = 0
        self.state = State.CLOSED


# Demo — a flaky downstream.
call_count = 0
def flaky_downstream():
    global call_count
    call_count += 1
    if call_count <= 5:
        raise ConnectionError("downstream down")
    return f"ok (call #{call_count})"


cb = CircuitBreaker(failure_threshold=3, cooldown_seconds=0.5)

# Three failures will open the circuit.
for i in range(6):
    try:
        result = cb.call(flaky_downstream)
        print(f"call {i+1}: {result} [state={cb.state.value}]")
    except Exception as e:
        print(f"call {i+1}: FAIL: {e} [state={cb.state.value}]")

print(f"\ndownstream was actually invoked {call_count} times "
      f"(of 6 attempted) — the rest were short-circuited.")

# After the cooldown, the next call probes — and now the downstream is healthy.
time.sleep(0.6)
print(f"\nafter cooldown, retrying...")
for i in range(3):
    try:
        result = cb.call(flaky_downstream)
        print(f"  result: {result} [state={cb.state.value}]")
    except Exception as e:
        print(f"  FAIL: {e}")


The circuit-breaker pattern usually ships as a library or sidecar concern, not application code. Resilience4j (Java), `pybreaker` (Python), Polly (.NET), and every service mesh (Istio, Linkerd) implement it. The library handles the state machine; the application just wraps calls in a `breaker.execute(...)` block.


## Bulkhead — isolating failure

Named after the watertight compartments in a ship's hull that prevent a single hole from sinking the whole vessel, the **bulkhead** pattern isolates resources so a failure in one part of the system doesn't drain resources from the rest.

**The motivating failure:**

A web server has a thread pool of 100 threads. Three downstream services — A, B, C — are called from different endpoints. Service A starts taking 10 seconds per request. Within minutes, all 100 threads are stuck waiting on A. Now endpoints that call B and C also fail, because there are no free threads. The slow service has consumed the *shared* resource and starved the others.

**The fix.** Allocate separate thread pools (or connection pools, or semaphores) per downstream — 30 threads for A, 30 for B, 30 for C, 10 reserved. Now A's slowness only consumes A's allotted threads. B and C continue serving. A's clients see slower responses (or timeouts); B's and C's clients are unaffected.

**Where bulkheads show up in practice:**

- **Thread / connection pools per dependency.** The most common form.
- **Separate process pools** for high-risk vs. low-risk operations (PDF generation in its own pool so a bad PDF doesn't OOM the API server).
- **Tenant isolation.** In a multi-tenant system, dedicate resources per tenant or per priority tier so one tenant's bug doesn't degrade everyone.
- **Async with bounded queues.** Each background-job type has its own queue; one job type's slowness doesn't back up the others.

**The cost.** Resources aren't fungible across bulkheads, so total utilization is lower. With one shared pool of 100 threads you could serve 100 fast requests at once; with three pools of 30 each, you can serve at most 30 of any one type even if the other pools are idle. That's the deliberate trade-off — slightly less efficient in the happy path, much more resilient in the unhappy path.

**Bulkhead + Circuit Breaker** is the standard pair: bulkhead isolates the resource damage, circuit breaker stops calling the broken downstream entirely. Together they prevent most cascading failures.


## Idempotency and safe retries

Networks drop packets. Services restart mid-request. Timeouts fire when the request actually succeeded. In a distributed system, **retries are inevitable** — and retries without care are how systems take themselves down.

**Three forces shape safe retry design:**

### Idempotency

A request is **idempotent** if making it twice has the same effect as making it once. `PUT /users/42/name = "Alice"` is idempotent — repeated execution leaves the same end state. `POST /orders/charge` is *not* idempotent by default — repeated execution charges the card twice.

For non-idempotent operations, the standard pattern is an **idempotency key**: the client generates a unique ID per logical operation; the server records the key alongside the result. On retry with the same key, the server returns the recorded result instead of re-executing. Stripe's API is the canonical example.

### Exponential backoff

If you retry immediately, you flood the recovering downstream and prevent it from catching up. **Exponential backoff** doubles the wait between retries: 100ms, 200ms, 400ms, 800ms, capped at some maximum. Recovering downstreams get a chance to actually recover.

### Jitter

Without jitter, every client retries at the same intervals — the "thundering herd" — slamming the downstream just as it tries to recover. **Add randomness:** sleep some random fraction of the backoff window. Full jitter (`sleep ~ uniform(0, backoff)`) is the standard recommendation; it spreads retries across the recovery window.

The combined recipe — **idempotent operation with exponential backoff and full jitter** — is the foundation under every cloud SDK, every well-built API client, and every retry library worth using.


### Retries with exponential backoff and jitter


In [ ]:
import random
import time


def with_retries(fn, max_attempts=5, base=0.1, cap=2.0):
    """Call fn; on exception, retry with exponential backoff + full jitter."""
    for attempt in range(max_attempts):
        try:
            return fn()
        except Exception as e:
            if attempt == max_attempts - 1:
                raise
            # exponential backoff: base * 2^attempt, capped
            backoff = min(cap, base * (2 ** attempt))
            # full jitter: sleep uniformly between 0 and backoff
            sleep = random.uniform(0, backoff)
            print(f"  attempt {attempt+1} failed ({e}); sleeping {sleep*1000:.0f}ms")
            time.sleep(sleep)
    raise RuntimeError("unreachable")


# Demonstrate the backoff schedule on a flaky function.
attempts = 0
def succeeds_on_fourth_try():
    global attempts
    attempts += 1
    if attempts < 4:
        raise ConnectionError("transient")
    return "success"


random.seed(42)   # for reproducible jitter
attempts = 0
result = with_retries(succeeds_on_fourth_try, max_attempts=5, base=0.05, cap=1.0)
print(f"\nfinal result: {result} (after {attempts} total attempts)")


# Show how full jitter spreads retries — with 100 clients all retrying,
# without jitter they'd all hit at the same time.
print(f"\nfull-jitter sleep samples (base=0.1s, attempt=3, so backoff=0.8s):")
backoff = 0.1 * (2 ** 3)   # 0.8s
samples = sorted(random.uniform(0, backoff) for _ in range(10))
for s in samples:
    print(f"  {s*1000:6.1f}ms  {'#' * int(s * 50)}")


**Don't retry everything.** Distinguish *transient* errors (timeouts, 503s, connection refused) from *permanent* ones (400 bad request, 404 not found, 403 forbidden). Retrying a 400 doesn't make it less 400; it just wastes resources. Build a retry policy that retries the transient class and surfaces the permanent class immediately.

**Watch out for retry storms across the stack.** If service A retries 5 times into service B, which retries 5 times into service C, a single C failure produces 25 calls to C from one user request. **Don't double-retry up the stack** — pick the layer at which retries happen and don't retry above it.


## Putting it together — the resilience stack

The resilience patterns combine into a standard recipe for every outbound service call in a microservices system:

```
   1. timeout         (give up before holding the connection too long)
   2. retry           (handle transient failures — with backoff and jitter)
   3. circuit breaker (stop calling a downstream that's broken)
   4. bulkhead        (isolate resources so failure doesn't spread)
   5. fallback        (return a degraded result, not a hard error)
```

A typical outbound call wraps in all five:

- **Timeout** fires at 200ms.
- **Retry** kicks in 2-3 times with exponential backoff + jitter for transient failures.
- **Circuit breaker** monitors the rolling failure rate; opens if >50% in last 10s; refuses calls for 30s.
- **Bulkhead** limits this dependency to its own pool of N threads/connections.
- **Fallback** returns a cached value, a default, or a graceful error message when the call fails.

In a service mesh, layers 1-4 live in the sidecar; the service just sees "this call failed" and applies the fallback. Without a mesh, a resilience library (Resilience4j, Polly, `tenacity`+`pybreaker`) provides the building blocks.

**The honest framing.** None of these patterns is exotic. Each is a small piece of code. The skill is *applying them consistently* across every outbound call in the system, and *not relying on them to mask design problems* — a circuit breaker around a downstream that fails 5% of the time hides a 5%-failure-rate downstream from your dashboards.


Notebook seven turns to **Operations & Reliability** — the day-to-day work that decides whether a well-designed system actually stays up. Observability (metrics, logs, traces) and the three pillars of "what is the system doing?". Deployment strategies (blue-green, canary, rolling) and how to ship without taking traffic down. Failure modes, graceful degradation, and load shedding. The patterns from this notebook tell you *what* to build; the patterns in notebook seven tell you *how to run it*.
